In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
from pathlib import Path
import shutil
import tempfile
from regulacion import calcular_hrp, calcular_cargo, cargar_parametros, cargo_estacion, validar_parametros, guardar_parametros
from modelado import asignar_estaciones

In [3]:
ruta_serie = Path("../data/processed/SERIE_TRABAJO_CTRELEUW.csv")
serie = pd.read_csv(ruta_serie, parse_dates=["MES"])
serie.info()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   MES                        93 non-null     datetime64[us]
 1   MAX_REAL_MW                93 non-null     float64       
 2   POTENCIA_DECLARADA_MW      93 non-null     float64       
 3   EDICION                    93 non-null     str           
 4   DESCARGA                   93 non-null     str           
 5   REVISADO                   93 non-null     bool          
 6   POSIBLE_COPIA_DECLARACION  93 non-null     bool          
 7   TEMPERATURA_MEDIA_C        93 non-null     float64       
 8   TEMPERATURA_MAX_C          93 non-null     float64       
 9   TEMPERATURA_MIN_C          93 non-null     float64       
dtypes: bool(2), datetime64[us](1), float64(5), str(2)
memory usage: 7.7 KB


In [4]:
parametros = cargar_parametros(Path("../data/parametros/PARAMETROS_REGULATORIOS.json"))
parametros

{'precio_ppad': 16,
 'porcentaje_piso': 90,
 'porcentaje_transitorio': 75,
 'estacion_transitoria': 'VERANO',
 'anio_transitorio': 2025,
 'kp_pico': 1.1,
 'kp_resto': 0.9,
 'meses_kp_pico': [12, 1, 2, 6, 7, 8],
 'fpunta': 1,
 'horas_hrp_por_dia': 18}

In [5]:
serie = asignar_estaciones(serie)

In [6]:
serie = calcular_hrp(serie, parametros=parametros)

In [7]:
serie = calcular_cargo(serie, parametros=parametros)

In [8]:
cargo_estacional = cargo_estacion(serie)
cargo_estacional

ESTACION  ANIO_ESTACION
INVIERNO  2019             1684893.74
          2020             1540243.04
          2021             1539452.51
          2022             1576942.53
          2023             1560198.24
          2024             1503904.32
          2025             1561211.80
          2026             1434626.73
VERANO    2018             1101936.42
          2019             1518004.17
          2020             1541275.05
          2021             1571411.63
          2022             1592856.35
          2023             1531327.97
          2024             1532244.20
          2025             1580441.24
Name: CARGO_USD, dtype: float64

In [9]:
parametros_prueba = parametros.copy()
parametros_prueba["fpunta"] = 0.5

In [10]:
ruta_serie = Path("../data/processed/SERIE_TRABAJO_CTRELEUW.csv")
serie_prueba = pd.read_csv(ruta_serie, parse_dates=["MES"])
serie_prueba.info()
 

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   MES                        93 non-null     datetime64[us]
 1   MAX_REAL_MW                93 non-null     float64       
 2   POTENCIA_DECLARADA_MW      93 non-null     float64       
 3   EDICION                    93 non-null     str           
 4   DESCARGA                   93 non-null     str           
 5   REVISADO                   93 non-null     bool          
 6   POSIBLE_COPIA_DECLARACION  93 non-null     bool          
 7   TEMPERATURA_MEDIA_C        93 non-null     float64       
 8   TEMPERATURA_MAX_C          93 non-null     float64       
 9   TEMPERATURA_MIN_C          93 non-null     float64       
dtypes: bool(2), datetime64[us](1), float64(5), str(2)
memory usage: 7.7 KB


In [11]:
serie_prueba = asignar_estaciones(serie_prueba)

In [12]:
serie_prueba = calcular_hrp(serie_prueba, parametros=parametros_prueba)

In [13]:
serie_prueba = calcular_cargo(serie_prueba, parametros=parametros_prueba)
serie_prueba[(serie_prueba["ESTACION"] == "VERANO") & (serie_prueba["ANIO_ESTACION"] == 2025)]

,MES,MAX_REAL_MW,POTENCIA_DECLARADA_MW,EDICION,DESCARGA,REVISADO,POSIBLE_COPIA_DECLARACION,TEMPERATURA_MEDIA_C,TEMPERATURA_MAX_C,TEMPERATURA_MIN_C,ESTACION,ANIO_ESTACION,DIAS_HABILES,HRP,KP,REQ_POT_MW,CARGO_USD
82,2025-11-01,38.628,36.081,2026-09,2026-10-20,False,False,17.526667,31.7,4.8,VERANO,2025,20,360,0.9,38.628,100123.78
83,2025-12-01,43.840,36.081,2026-09,2026-10-20,False,False,21.383871,34.9,8.2,VERANO,2025,23,414,1.1,43.840,159717.89
84,2026-01-01,44.244,36.081,2026-09,2026-10-20,False,False,21.735484,37.9,10.6,VERANO,2025,22,396,1.1,44.244,154181.49
85,2026-02-01,44.132,36.081,2026-09,2026-10-20,False,False,20.885714,36.8,6.5,VERANO,2025,20,360,1.1,44.132,139810.18
86,2026-03-01,43.560,36.081,2026-09,2026-10-20,False,False,18.964516,33.3,5.5,VERANO,2025,22,396,0.9,43.560,124198.27
87,2026-04-01,39.348,36.081,2026-09,2026-10-20,False,False,14.293333,31.8,1.0,VERANO,2025,22,396,0.9,39.348,112189.02


In [14]:
parametros_invalidos = parametros.copy()
parametros_invalidos["porcentaje_piso"] = 150
parametros_invalidos["estacion_transitoria"] = "OTOÑO"
parametros_invalidos["kp_pico"] = -100
parametros_invalidos["fpunta"] = -1

try:
    validar_parametros(parametros_invalidos)
except ValueError as error:
    print("Error esperado:", error)

Error esperado: El parámetro kp_pico tiene que ser mayor que 0 y vale -100
El parámetro fpunta tiene que ser mayor que 0 y vale -1
El parámetro porcentaje_piso tiene que estar entre 0 y 100, y vale 150
estacion_transitoria tiene que ser VERANO o INVIERNO, y es OTOÑO


In [15]:
validar_parametros(parametros)

In [16]:
with tempfile.TemporaryDirectory() as carpeta:
    copia = Path(carpeta) / "PARAMETROS_PRUEBA.json"
    shutil.copy("../data/parametros/PARAMETROS_REGULATORIOS.json", copia)

    guardar_parametros(copia, {"precio_ppad": {"valor": 18, "norma": "Norma de prueba", "vigencia_desde": "2027-05-01"}})
    print(cargar_parametros(copia)["precio_ppad"])

    try:
        guardar_parametros(copia, {"anio_transitorio": {"valor": 20026, "norma": "Res. SE 22/2026, art. 6", "vigencia_desde": "2026-01-28"}})
    except ValueError as error:
        print(error)
    print(cargar_parametros(copia)["anio_transitorio"])

18
El parámetro anio_transitorio tiene que ser un año de cuatro cifras entre 2000 y 2100, y vale 20026
2025
